<a href="https://colab.research.google.com/github/hzeattar/BLM-02BTC-Search/blob/master/BLM_02BTC_GPU_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLM 0.2 BTC Puzzle - GPU/CPU Search on Google Colab

Target: `1KfZGvwZxsvSmemoCmEV75uqcNzYBHjkHZ`

Path: `m/44'/0'/0'/0/0` (Legacy P2PKH)

## Setup
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells.

In [ ]:
# Clone repo
!git clone https://github.com/hzeattar/BLM-02BTC-Search.git
%cd BLM-02BTC-Search
!pip install -q bip_utils
!pip install -q mnemonic # Added this line to install the missing library
!apt-get update -qq && apt-get install -y -qq hashcat > /dev/null 2>&1
!hashcat --version

Cloning into 'BLM-02BTC-Search'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 26 (delta 11), reused 15 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 15.25 KiB | 15.25 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/BLM-02BTC-Search/BLM-02BTC-Search/BLM-02BTC-Search
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
v6.2.5


In [ ]:
import os
import sys
import time
import hashlib
import hmac
from itertools import permutations, product, islice
from multiprocessing import Pool, cpu_count, Manager

from bip_utils import (
    Bip39MnemonicValidator, Bip39SeedGenerator, Bip39Languages,
    Bip44, Bip44Coins, Bip44Changes
)
from mnemonic import Mnemonic

TARGET = '1KfZGvwZxsvSmemoCmEV75uqcNzYBHjkHZ'

PASSPHRASES = [
    '', 'BREATHE', 'breathe', 'TUESDAY', 'tuesday',
    'BREATHE TUESDAY', 'breathe tuesday',
    "I CAN'T BREATHE", 'I CANT BREATHE', 'ICANTBREATHE',
    'BLACK LIVES MATTER', 'BLM', 'ONLY REAL BITCOIN', 'ONLY BITCOIN',
    'ORDER AND STABILITY', 'THIS IS THE FIRST PREDICTION',
    'WELCOME TO THE BRAVE NEW WORLD', 'PAY FOR THE FUTURE',
    'FUCK THIS SHIT', 'RERUM COGNOSCERE CAUSAS',
    'FIAT JUSTITIA ET PEREAT MUNDUS', 'UBI BENE IBI PATRIA',
    'George Floyd', 'GEORGE FLOYD', '2000', '0.2',
    'SATOSHI NAKAMOTO', 'satoshi nakamoto', 'BITCOIN', 'bitcoin',
    'X', 'SUN', 'MOON', 'TOWER',
    '1KfZGvwZxsvSmemoCmEV75uqcNzYBHjkHZ',
]

mnemo = Mnemonic('english')
wl = mnemo.wordlist
validator = Bip39MnemonicValidator(Bip39Languages.ENGLISH)

def derive(words, passphrase=''):
    phrase = ' '.join(words)
    seed = Bip39SeedGenerator(phrase, Bip39Languages.ENGLISH).Generate(passphrase)
    ctx = Bip44.FromSeed(seed, Bip44Coins.BITCOIN).Purpose().Coin().Account(0).Change(Bip44Changes.CHAIN_EXT).AddressIndex(0)
    return ctx.PublicKey().ToAddress()

def check(words_tuple):
    if not validator.IsValid(' '.join(words_tuple)):
        return None
    for pp in PASSPHRASES:
        try:
            if derive(words_tuple, pp) == TARGET:
                return (' '.join(words_tuple), pp)
        except Exception:
            pass
    return None

print('Setup complete. CPU cores:', cpu_count())

BLM 0.2 BTC PUZZLE SOLVER - COMPREHENSIVE FINAL EDITION
Target: 1KfZGvwZxsvSmemoCmEV75uqcNzYBHjkHZ
CPU Cores: 2
BIP39 words loaded: 2048

[*] PHASE 1: Original template + passphrases...
    Tested: 4,100,000 | Valid: 256,250 | Time: 439.9s
    Phase 1 complete. Tested 4,194,304, found 262,144 valid phrases.

[*] PHASE 2: Fuzzy word replacements...
    Testing 33 single-word replacements...
    Replacement 4:read | 2,000,000

In [ ]:
# Strategy 1: Extended matrix from railway_search.py
MATRIX = [
    ['subject', 'base', 'model'],
    ['aware'],
    ['all', 'tower'],
    ['decide'],
    ['first', 'arrive'],
    ['this', 'trust'],
    ['party', 'must'],
    ['public'],
    ['announce'],
    ['need', 'system', 'food'],
    ['agree', 'food'],
    ['order', 'history', 'black'],
]

total = 1
for p in MATRIX:
    total *= len(p)
print(f'Matrix combinations: {total:,}')

from multiprocessing import Pool

t0 = time.time()
found = None
with Pool(processes=cpu_count()) as pool:
    for i, result in enumerate(pool.imap_unordered(check, product(*MATRIX), chunksize=100)):
        if result:
            found = result
            break
        if (i+1) % 1000 == 0:
            print(f'Tested {i+1:,} / {total:,} in {time.time()-t0:.1f}s')

if found:
    print('MATCH FOUND:', found)
else:
    print('No match in matrix.')

Matrix combinations: 864
No match in matrix.


In [ ]:
import math
import time
from multiprocessing import Pool, cpu_count

STRONGEST = ['base', 'aware', 'all', 'decide', 'first', 'this',
             'party', 'public', 'announce', 'system', 'agree', 'order']

def unrank_perm(index, elements):
    elements = list(elements)
    n = len(elements)
    result = []
    for i in range(n, 0, -1):
        fact = math.factorial(i-1)
        j, index = divmod(index, fact)
        result.append(elements.pop(j))
    return tuple(result)

TOTAL_PERM = math.factorial(12)
# --- عدل هذا الرقم بناءً على آخر نتيجة ظهرت لك ---
START_INDEX = 0
LIMIT = 50_000_000

print(f'12! = {TOTAL_PERM:,}')
print(f'Resuming from: {START_INDEX:,} up to {START_INDEX + LIMIT:,}')

t0 = time.time()
found = None

# Using pool to check range(START_INDEX, START_INDEX + LIMIT)
with Pool(processes=cpu_count()) as pool:
    task_generator = (unrank_perm(k, STRONGEST) for k in range(START_INDEX, START_INDEX + LIMIT))
    for i, result in enumerate(pool.imap_unordered(check, task_generator, chunksize=5000)):
        current_idx = START_INDEX + i
        if result:
            found = result
            break
        if (i + 1) % 100000 == 0:
            elapsed = time.time() - t0
            print(f'Tested {current_idx + 1:,} / {START_INDEX + LIMIT:,} in {elapsed:.1f}s')

if found:
    print('MATCH FOUND:', found)
else:
    print(f'No match in the range {START_INDEX:,} to {START_INDEX + LIMIT:,}.')

In [ ]:
# Strategy 3: Mixed visual/index set permutations
MIXED = ['subject', 'tower', 'food', 'black', 'moon', 'dose',
         'mean', 'trouble', 'wise', 'real', 'this', 'order']

t0 = time.time()
found = None
with Pool(processes=cpu_count()) as pool:
    for i, result in enumerate(pool.imap_unordered(check, (unrank_perm(k, MIXED) for k in range(LIMIT)), chunksize=5000)):
        if result:
            found = result
            break
        if (i+1) % 100000 == 0:
            print(f'Tested {i+1:,} / {LIMIT:,} in {time.time()-t0:.1f}s')

if found:
    print('MATCH FOUND:', found)
else:
    print(f'No match in first {LIMIT:,} permutations.')